<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Elena/ZeroShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 180 (delta 66), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (180/180), 1.52 MiB | 9.34 MiB/s, done.
Resolving deltas: 100% (99/99), done.


#Department - pretrained model "facebook/bart-large-mnli"

In [2]:
# Import pretrained model "facebook/bart-large-mnli".
# This is the checkpoint for bart-large after being trained on the MultiNLI (MNLI) dataset.
# The Multi-Genre Natural Language Inference (MultiNLI) corpus is a crowd-sourced collection of 433k sentence pairs annotated with textual entailment information. The corpus is modeled on the SNLI corpus, but differs in that covers a range of genres of spoken and written text, and supports a distinctive cross-genre generalization evaluation.
from transformers import pipeline
pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [3]:
import pandas as pd
df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
# Predict department with a pretrained model based on position name
dpt_labels = df["department"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = dpt_labels
result = pipe(text, candidate_labels)

In [4]:
prediction = []

for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

In [5]:
from sklearn.metrics import classification_report
print(classification_report(df["department"], prediction))

                        precision    recall  f1-score   support

        Administrative       0.09      0.62      0.15        84
  Business Development       0.48      0.53      0.50        78
            Consulting       0.24      0.81      0.37       195
      Customer Support       0.62      0.52      0.57        48
       Human Resources       0.57      0.67      0.61        69
Information Technology       0.73      0.50      0.59       309
             Marketing       0.59      0.49      0.53       133
                 Other       0.75      0.20      0.31      1235
    Project Management       0.75      0.65      0.70       173
            Purchasing       0.15      0.51      0.23        72
                 Sales       0.74      0.37      0.50       219

              accuracy                           0.39      2615
             macro avg       0.52      0.53      0.46      2615
          weighted avg       0.65      0.39      0.41      2615



Precision measures the accuracy of positive predictions (True Positives / (True Positives + False Positives)), answering "Of all predicted positives, how many were actually positive?".
while Recall measures the model's ability to find all relevant instances (True Positives / (True Positives + False Negatives)), answering "Of all actual positives, how many did we find?"
Without any training relevant to our specific task, the total accuracy is around 40%.

#Seniority - pretrained model  "facebook/bart-large-mnli"


In [6]:
# Predict seniority with a pretrained model

In [7]:
sen_labels = df["seniority"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = sen_labels
result = pipe(text, candidate_labels)

In [8]:
prediction = []

for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

In [9]:
from sklearn.metrics import classification_report
print(classification_report(df["seniority"], prediction))

              precision    recall  f1-score   support

    Director       0.76      0.84      0.79       141
      Junior       0.89      0.32      0.47       230
        Lead       0.53      0.15      0.24       455
  Management       0.31      0.28      0.29       408
Professional       0.63      0.77      0.70      1211
      Senior       0.38      0.88      0.53       170

    accuracy                           0.56      2615
   macro avg       0.58      0.54      0.50      2615
weighted avg       0.58      0.56      0.53      2615



The overall arrcuray for predicting the seniority of the CVs is at 56%

In [10]:
# Train the model with our own data to improve accuracy.

In [11]:
# Department
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
labels = le.fit_transform(df["department"].drop_duplicates())

In [12]:
dpt = le.inverse_transform([ 7,  5,  3,  2,  8, 10,  1,  0,  6,  4,  9])

In [13]:
mapping = dict(zip(dpt, labels))

In [14]:
dataset = df[["position", "department"]]

In [15]:
dataset["labels"] = df["department"].map(mapping)

/tmp/ipython-input-317583838.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["labels"] = df["department"].map(mapping)


In [16]:
dataset = dataset.drop(columns = "department")

In [17]:
from datasets import Dataset

In [18]:
hf_dataset = Dataset.from_pandas(dataset)
split_datasets = hf_dataset.train_test_split(test_size = 0.2, seed = 42)

train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

In [30]:
from transformers import AutoTokenizer

model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
  return tokenizer(batch["position"], truncation=True, padding = "max_length", max_length = 106)

train_dataset = train_dataset.map(tokenize, batched = True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = eval_dataset.map(tokenize, batched = True)
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/2092 [00:00<?, ? examples/s]

Map:   0%|          | 0/2092 [00:00<?, ? examples/s]

In [31]:
train_dataset

Dataset({
    features: ['position', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2092
})

In [32]:
 id2label = {int(k):str(v) for k,v in zip(labels, dpt)}
 label2id = {str(v):int(k) for k,v in id2label.items()}

In [33]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(label2id),
    id2label=id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True
)

Some weights of BartForSequenceClassification were not initialized from the model checkpoint at facebook/bart-large-mnli and are newly initialized because the shapes did not match:
- classification_head.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([11]) in the model instantiated
- classification_head.out_proj.weight: found shape torch.Size([3, 1024]) in the checkpoint and torch.Size([11, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
# Train the model
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    "test_trainer", report_to="none"
)


In [26]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [35]:
import numpy as np
import evaluate


metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [36]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    compute_metrics = compute_metrics
)

trainer.evaluate()

{'eval_loss': 2.5074105262756348,
 'eval_model_preparation_time': 0.0083,
 'eval_accuracy': 0.05353728489483748,
 'eval_runtime': 57.8394,
 'eval_samples_per_second': 36.169,
 'eval_steps_per_second': 4.53}